In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from astropy.table import Table
from pathlib import Path
import numpy as np
import concurrent.futures
from astropy.cosmology import Planck18
from astropy.io import ascii
import random
from astropy.coordinates import SkyCoord
from astropy import units as u
import networkx as nx
import pandas as pd
import requests
import gc, os
from astropy.io import fits

In [ ]:
filt_n1 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt1.ecsv")
filt_n2 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt2.ecsv")

In [ ]:
def cartesianas(data):
    data['distance'] = Planck18.comoving_distance(data['Z'])

    ra_values = np.array(data['RA'])
    dec_values = np.array(data['DEC'])

    coords = SkyCoord(
        ra = ra_values * u.deg,
        dec = dec_values * u.deg,
        distance = data['distance'],
        frame = 'icrs'
    )
    data['x'] = coords.cartesian.x.value
    data['y'] = coords.cartesian.y.value
    data['z'] = coords.cartesian.z.value

In [ ]:
def url(seed_for_this_file):
    urls = [
        f"https://data.desi.lbl.gov/public/dr1/survey/catalogs/dr1/LSS/iron/LSScats/v1.5/LRG_NGC_{i}_clustering.ran.fits"
        for i in range(18)
    ]
    rng_local = np.random.default_rng(seed_for_this_file)
    chosen_url = rng_local.choice(urls)
    rand = Table.read(chosen_url)
    return rand

def process_one(i, number, data, base_seed):
    print(i)

    seed_i = base_seed + i
    rng = np.random.default_rng(seed_i)

    rand = url(seed_i)

    if number == 1:
        mask = (rand['RA'] >= 110) & (rand['RA'] <= 260) & (rand['DEC'] >= -10) & (rand['DEC'] <= 8)
    elif number == 2:
        mask = (rand['RA'] >= 180) & (rand['RA'] <= 260) & (rand['DEC'] >= 30) & (rand['DEC'] <= 40)
    else:
        raise ValueError("Orientation must be 1 or 2.")

    rand = rand[mask]

    idx_random = rng.choice(len(rand), size=len(data), replace=False)
    rand_subset = rand[idx_random]

    cartesianas(rand_subset)
    rand_subset = rand_subset[['TARGETID', 'x', 'y', 'z']]
    rand_subset['type'] = 'rand'

    filename_rand = f"/content/drive/MyDrive/DESI/rand/LRG_NGC_{number}_random_{i}_filt.fits.gz"
    rand_subset.write(filename_rand, overwrite=True)

    del rand_subset
    gc.collect()
    return i

def save_file_parallel(data, number, n_random=100, seed=42, max_workers=4):
    Path("/content/drive/MyDrive/DESI/rand").mkdir(exist_ok=True)

    indices = list(range(n_random))
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one, i, number, data, seed): i for i in indices}
        for future in concurrent.futures.as_completed(futures):
            i = futures[future]
            try:
                _ = future.result()
                print(f"✓ Archivo {i} listo")
            except Exception as e:
                print(f"⚠ Error en {i}: {e}")

    print("Files saved")

In [ ]:
%%time
save_file_parallel(filt_n1, number=1, n_random=100, seed=42, max_workers=4)

In [ ]:
%%time
save_file_parallel(filt_n1, number=2, n_random=100, seed=42, max_workers=4)